# 04 - Heatmap Prototype
Step-by-step construction of the MTBF spatial heatmap.

In [ ]:
import sys
sys.path.insert(0, "..")
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pandas as pd

metrics = pd.read_csv("../data/processed/mtbf_metrics.csv")
with open("../data/layout/zones.json") as f:
    zones = json.load(f)
scores = dict(zip(metrics["machine_id"], metrics["health_score"]))
print("Loaded", len(zones["machines"]), "machines")

## Step 1: Color-mapping function

In [ ]:
def health_color(score):
    if score >= 70: return "#2ECC71"
    if score >= 40: return "#F39C12"
    return "#E74C3C"

for s in [90, 60, 25]:
    print(f"  Score {s:3d} -> {health_color(s)}")

## Step 2: Blank canvas

In [ ]:
fig, ax = plt.subplots(figsize=(14, 9.35))
ax.set_xlim(0, 30)
ax.set_ylim(0, 20)
ax.set_aspect("equal")
ax.set_facecolor("#E8E8E8")
ax.add_patch(Rectangle((0,0), 30, 20, facecolor="#F0F0F0",
    edgecolor="#0D1B2A", linewidth=2))
plt.title("Step 2: Blank canvas (30x20)")
plt.tight_layout()

## Step 3: Area bands + aisles

In [ ]:
area_colors = {"Machining": "#EAF2FB", "Painting": "#EBF5FB", "Assembly": "#E8F8F5"}
for aname, az in zones["areas"].items():
    ax.add_patch(Rectangle((0, az["y_min"]), 30, az["y_max"]-az["y_min"],
        facecolor=area_colors[aname], edgecolor="#BBB", linewidth=0.8, alpha=0.5))
    ax.text(-0.1, (az["y_min"]+az["y_max"])/2, aname,
        va="center", ha="right", rotation=90, fontsize=8)
for aisle in zones["aisles"]:
    ax.add_patch(Rectangle((aisle["x"], 0), aisle["w"], 20,
        facecolor="#CCCCCC", alpha=0.6))
plt.title("Step 3: Area bands and aisles")
plt.tight_layout()

## Step 4: Machine health boxes

In [ ]:
for mid, mz in zones["machines"].items():
    score = scores.get(mid, 50.0)
    color = health_color(score)
    ax.add_patch(Rectangle((mz["x"], mz["y"]), mz["w"], mz["h"],
        facecolor=color, edgecolor="#0D1B2A", linewidth=1.1, alpha=0.87))
    cx = mz["x"] + mz["w"]/2
    cy = mz["y"] + mz["h"]/2
    ax.text(cx, cy+0.25, mid, ha="center", va="center",
        fontsize=4.5, fontweight="bold", color="#0D1B2A")
    ax.text(cx, cy-0.15, f"{score:.0f}", ha="center", va="center",
        fontsize=7.5, fontweight="bold", color="#0D1B2A")
plt.title("Step 4: Health heatmap complete")
plt.tight_layout()
fig.savefig("../outputs/figures/prototype_heatmap.png", dpi=100, bbox_inches="tight")
print("Prototype saved to outputs/figures/prototype_heatmap.png")

## Step 5: Production-quality heatmap

In [ ]:
from src.spatial.heatmap import build_and_save
path = build_and_save()
print(f"Production heatmap -> {path}")